In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from reprojection import *
from show import *
from utils import *
import pandas as pd
from datetime import datetime
import fnmatch


qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in ""


In [2]:
def make_map(files, pole='north', stonyhurst=False, q=-30, mu_thr=0.1):
    s = np.load('/home/ulyanov/data/solo/phi/distortion/fdt/distortion.npz')
    xd, yd = s['xd'], s['yd']

    df = pd.read_csv('/home/ulyanov/data/solo/phi/wcs/fdt/disk_centers.csv', skipinitialspace=True).dropna()
    dids = df['did'].to_numpy()
    xu_sun, yu_sun, ru_sun = df['xu_sun'].to_numpy(), df['yu_sun'].to_numpy(), df['ru_sun'].to_numpy()


    nx, ny = 1024, 1024
    xc, yc = 511.5, 511.5
    Rsun = 480


    if pole == 'north':
        view_new = View(nx, ny, xc, yc, Rsun, -90, 90, 0, rsun_arc=q * 3600) ### North pole view
    else:
        view_new = View(nx, ny, xc, yc, Rsun, 90, -90, 0, rsun_arc=q * 3600) ### South pole view

    grid = view_new.grid()
    mean, variance, coverage = np.zeros_like(grid[0]), np.zeros_like(grid[0]), np.zeros_like(grid[0])

    for file in files[:]:
        with fits.open(file) as hdul:
            header = hdul[0].header.copy()
            data = hdul[0].data.copy()

        did = int(file.split('_')[-1].split('.')[0])
        data = undistort(data, header, xd, yd)

        view = View.from_header(header)
        view.update(xc=xu_sun[dids == did][0], yc=yu_sun[dids == did][0], crota=view.crota + 0.25, rsun=ru_sun[dids == did][0], inplace=True)

        transform = (view_new.to_carrington(correct_mu=True) -
                     view.to_synoptic(correct_mu=True, correct_dr=True, mu_thr=mu_thr, stonyhurst=stonyhurst))
        grid_, alpha = transform(grid)
        data = bilinear(data, *grid_) * alpha

        n = (~np.isnan(data)) * (1 / alpha) ** 4
        coverage += np.nan_to_num(n)
        mean_ = mean.copy()
        mean += np.nan_to_num((data - mean) * n / coverage)
        variance += np.nan_to_num((data - mean) * (data - mean_) * n)


    variance = variance / coverage
    mean[coverage == 0] = np.nan
    coverage[coverage == 0] = np.nan
    variance[coverage == 0] = np.nan

    return mean, view_new

In [3]:
df = pd.read_csv('/home/ulyanov/data/solo/phi/wcs/fdt/wcs.csv', skipinitialspace=True).dropna()

dates = np.array([datetime.fromisoformat(date) for date in df['date']])
car_rot = df['car_rot'].to_numpy()
hgln = df['hgln_obs'].to_numpy()

np.unique(car_rot)

array([2279, 2280, 2281, 2282, 2283, 2284, 2285, 2286, 2287, 2288, 2289,
       2290, 2291, 2292, 2293, 2294, 2295, 2296, 2297, 2298, 2299, 2300,
       2301, 2302, 2303, 2304, 2305, 2306])

In [4]:
t = np.where(car_rot == 2302)[0]

files = np.array(sorted(glob.glob('/home/ulyanov/data/solo/phi/2024_2025/*')))
files = files[t]

In [8]:
mean, view = make_map(files, pole='north')

/tmp/ipykernel_33276/3676298927.py:46: RuntimeWarning: invalid value encountered in divide
  variance = variance / coverage


In [9]:
show_data(mean, view, title='North', label=r'$B_{los}$')

In [7]:
files = sorted(glob.glob('/home/ulyanov/data/solo/phi/2024_2025/*'))
t = np.where(np.all([dates > datetime(2025,8,29), dates < datetime(2025,10,8)], axis=0))[0]
files = np.array(files)[t]


len(files)

122

In [10]:
mean, view = make_map(files, pole='south', q=0)

/tmp/ipykernel_75893/3676298927.py:46: RuntimeWarning: invalid value encountered in divide
  variance = variance / coverage


In [11]:
show_data(mean, view, label=r'$B_{los}$')

In [9]:
files_ = sorted(glob.glob('/home/ulyanov/data/solo/phi/2024_2025/*'))



files = []
for file in files_:
    if fnmatch.fnmatch(file, '*blos_202411*'):
        files.append(file)

print(len(files))
print(files[0], files[-1])

220
/home/ulyanov/data/solo/phi/2024_2025/solo_L2_phi-fdt-blos_20241101T001503_V202602220112_0451010501.fits.gz /home/ulyanov/data/solo/phi/2024_2025/solo_L2_phi-fdt-blos_20241130T211503_V202602220151_0451300508.fits.gz


In [4]:
for i in np.unique(car_rot):

    t = np.where(car_rot == i)[0]

    files = np.array(sorted(glob.glob('/home/ulyanov/data/solo/phi/2024_2025/*')))
    files = files[t]

    mean, view = make_map(files, pole='south')
    show_data(mean, view, title=f'{i}', label=r'$B_{los}$', to_file=f'temp1/{i}.png')


/tmp/ipykernel_70354/1956693171.py:46: RuntimeWarning: invalid value encountered in divide
  variance = variance / coverage
/tmp/ipykernel_70354/1956693171.py:46: RuntimeWarning: invalid value encountered in divide
  variance = variance / coverage
/tmp/ipykernel_70354/1956693171.py:46: RuntimeWarning: invalid value encountered in divide
  variance = variance / coverage
/tmp/ipykernel_70354/1956693171.py:46: RuntimeWarning: invalid value encountered in divide
  variance = variance / coverage
/tmp/ipykernel_70354/1956693171.py:46: RuntimeWarning: invalid value encountered in divide
  variance = variance / coverage
/tmp/ipykernel_70354/1956693171.py:46: RuntimeWarning: invalid value encountered in divide
  variance = variance / coverage
/tmp/ipykernel_70354/1956693171.py:46: RuntimeWarning: invalid value encountered in divide
  variance = variance / coverage
/tmp/ipykernel_70354/1956693171.py:46: RuntimeWarning: invalid value encountered in divide
  variance = variance / coverage
/tmp/ipy

/tmp/ipykernel_67206/1516524003.py:46: RuntimeWarning: invalid value encountered in divide
  variance = variance / coverage
